In [1]:
!git clone https://github.com/AIVIETNAM-AIO-ERICPHAM73/hydraulic-machine-health.git

Cloning into 'hydraulic-machine-health'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 50 (delta 36), reused 14 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 6.48 KiB | 6.48 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
from pathlib import Path
ROOT_DRIVE_DIR = Path('/content/drive/MyDrive/AIO/AIO_Module3_Hydraulic')
RAW_DATA_DRIVE_DIR = ROOT_DRIVE_DIR / 'data_raw'
PROCESSED_DATA_DRIVE_DIR = ROOT_DRIVE_DIR / 'data_processed'
ARTIFACTS_DRIVE_DIR = ROOT_DRIVE_DIR / 'artifacts'
for p in [RAW_DATA_DRIVE_DIR, PROCESSED_DATA_DRIVE_DIR, ARTIFACTS_DRIVE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [16]:
#%%writefile hydraulic-machine-health/src/config.py
from pathlib import Path
from typing import Final

import sys, os

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------

RANDOM_STATE: Final[int] = 42

# ---------------------------------------------------------------------------
# Dataset contract
# ---------------------------------------------------------------------------

EXPECTED_CYCLES: Final[int] = 2205
CYCLE_DURATION_SECONDS: Final[int] = 60

PROFILE_COLUMNS: Final[list[str]] = ['cooler_condition', 'valve_condition', 'internal_pump_leakage', 'hydraulic_accumulator', 'stable_flag']

TARGET_COLUMN: Final[str] = 'pump_leakage'
META_COLUMNS: Final[list[str]] = ['cooler_condition', 'valve_condition', 'hydraulic_accumulator', 'stable_flag']

PUMP_LABELS: Final[dict[int, str]] = {0: 'no_leakage', 1: 'weak_leakage', 2: 'severe_leakage'}

EXCLUDED_MODEL_COLUMNS: Final[list[str]] = ['cooler_condition', 'valve_condition', 'hydraulic_accumulator', 'stable_flag', 'cycle_id']

# ---------------------------------------------------------------------------
# Sensor metadata
# ---------------------------------------------------------------------------

SENSORS: Final[list[str]] = [
    'PS1', 'PS2', 'PS3', 'PS4', 'PS5', 'PS6',
    'EPS1',
    'FS1', 'FS2',
    'TS1', 'TS2', 'TS3', 'TS4',
    'VS1',
    'CE', 'CP', 'SE'
]

SENSOR_SAMPLING_HZ: Final[dict[str, int]] = {
    'PS1': 100, 'PS2': 100, 'PS3': 100, 'PS4': 100, 'PS5': 100, 'PS6': 100,
    'EPS1': 100,
    'FS1': 10, 'FS2': 10,
    'TS1': 1, 'TS2': 1, 'TS3': 1, 'TS4': 1,
    'VS1': 1,
    'CE': 1, 'CP': 1, 'SE': 1
}

# Expected time points per cycle = sampling rate(Hz) * 60(seconds)
EXPECTED_TIMEPOINTS: Final[dict[str, int]] = {
    sensor: hz * CYCLE_DURATION_SECONDS for sensor, hz in SENSOR_SAMPLING_HZ.items()
}


# ---------------------------------------------------------------------------
# Feature engineering configuration
# ---------------------------------------------------------------------------

START_END_FRACTION: Final[float] = 0.10
N_SEGMENTS_V2: Final[int] = 6


# ---------------------------------------------------------------------------
# Validation / research configuration
# ---------------------------------------------------------------------------

N_SPLITS: Final[int] = 5
BLOCKED_HOLDOUT_FRACTION: Final[float] = 0.20
NOISE_SNR_DB_LEVELS: Final[tuple[int, ...]] = (30, 20, 10)
SHAP_TOP_K: Final[int] = 10


# ---------------------------------------------------------------------------
# Path convention
# ---------------------------------------------------------------------------

def get_project_root() -> Path:
    # Check for running on Colab or VS Code
    if 'google.colab' in sys.modules:
        # If Colab
        notebook_path = Path('/content/hydraulic-machine-health/notebooks/')
        project_root_path = notebook_path.parent.parent
    elif '__vsc_ipynb_file__' in globals():
        # If VS Code
        notebook_path = Path(globals()['__vsc_ipynb_file__'])
        project_root_path = notebook_path.parent.parent
    else:
        # Backup for normal Jupyter Notebook or file.py
        try:
            notebook_path = Path(__file__).resolve()
            project_root_path = notebook_path.parent.parent
        except NameError:
            # If cannot found, get the current working folder
            project_root_path = Path(os.getcwd())

    # Trace back to the root folder and config sys.path
    os.chdir(project_root_path)

    if str(project_root_path) not in sys.path:
        sys.path.append(str(project_root_path))

    return project_root_path


In [17]:
#%%writefile hydraulic-machine-health/src/data.py
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

ROOT_DIR = get_project_root()

from src.config import (EXPECTED_CYCLES, EXPECTED_TIMEPOINTS, META_COLUMNS, PROFILE_COLUMNS, TARGET_COLUMN)

def load_profile(profile_path: Path) -> pd.DataFrame:
    # Read profile.txt as a tab-delimited file with no header
    profile_arr = np.loadtxt(RAW_DATA_DRIVE_DIR/ 'profile.txt', delimiter='\t', dtype=np.int32)

    # Assign PROFILE_COLUMNS in the documented order
    profile_df = pd.DataFrame(profile_arr, columns=PROFILE_COLUMNS, dtype=np.int32)

    # Create a cycle_id index from row order
    profile_df['cycle_id'] = profile_df.index

    # Preserve all five profile columns for audit and sensitivity analysis
    return profile_df

ModuleNotFoundError: No module named 'src'